# Kombinierte Labels: 1D-CNN

**Hypothese:** Lokale Muster benachbarter Wellenlängen ergänzen die tabellarische Sicht und verbessern die kombinierte BAcc gegenüber der RBF-SVM.

Die CNN erhält zwei geordnete Kanäle: skalierte Reflexionen und normierte Wellenlängenpositionen. AEZ und Month bleiben ein separater Kontextzweig. Die 23 kombinierten Klassen garantieren gültige Crop/Stage-Paare.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_spectral_cnn
from awp2.preprocessing import PreprocessingConfig

COMBINED_SVM_CV_BACC = 0.834
COMBINED_SVM_VAL_BACC = 0.845

cnn_preprocessing = PreprocessingConfig(scale=True)

## 1. Hyperparameter suchen

Die Suche variiert Filterzahl und lokale Kernelbreite in vier kleinen CNNs. Jede CNN nutzt Early Stopping auf einem inneren Trainingsanteil; die fünf gemeinsamen Folds entscheiden ausschließlich nach `bacc_combined`.

In [ ]:
tuned = tune(
    make_combined_spectral_cnn(),
    TuneConfig(
        name="combined_cnn_search",
        approach="combined",
        description=(
            "Kleine 1D-CNN auf kombinierten Crop/Stage-Labels: "
            "Filterzahl und Kernelbreite vergleichen."
        ),
        preprocessing=cnn_preprocessing,
        param_grid={
            "estimator__filters": [8, 16],
            "estimator__kernel_size": [5, 9],
            "estimator__max_epochs": [30],
        },
    ),
)

display(tuned.candidates.sort_values("rank"))
{
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "tuning_run": tuned.run_id,
}

## 2. Einmal validieren

Die CNN bleibt wegen ihres deutlich niedrigeren CV-Scores keine Kandidatin für weitere Architekturvarianten. Der vorab festgelegte CV-Gewinner wird einmal auf der gemeinsamen Validierung eingeordnet; daraus folgt keine weitere Suche.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="combined_cnn",
        approach="combined",
        description=(
            "Beste kleine 1D-CNN aus combined_cnn_search, einmal auf der "
            "Validierung bewertet."
        ),
        preprocessing=cnn_preprocessing,
        tuning_run=tuned.run_id,
        log_model=False,
    ),
)

result.metrics.model_dump()

## 3. Analyse

Die Vergleichstabelle zeigt den Abstand zur tabellarischen Referenz. Die Abbildungen dokumentieren Kandidatenstreuung, Klassenrecall und die häufigsten kombinierten Verwechslungen.

In [ ]:
from awp2.data import combined_label
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

comparison = pd.DataFrame(
    {
        "CV-BAcc kombiniert": {
            "1D-CNN": tuned.best_score,
            "RBF-SVM": COMBINED_SVM_CV_BACC,
        },
        "Validierungs-BAcc kombiniert": {
            "1D-CNN": result.metrics.bacc_combined,
            "RBF-SVM": COMBINED_SVM_VAL_BACC,
        },
    }
)
display(comparison)

top_confusions = (
    pd.DataFrame(
        {
            "Wahr": combined_label(result.y_val),
            "Vorhergesagt": combined_label(result.y_pred),
        }
    )
    .query("Wahr != Vorhergesagt")
    .value_counts()
    .head(10)
)
display(top_confusions)

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__filters",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title("Kombinierte Labels: CV-Score der 1D-CNN")
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

save_doc_figure(tuning_ax.figure, "combined_cnn_tuning", MODEL_DOCS_FIGURES_DIR)

confusion_figure = plot_confusion_matrices(result.y_val, result.y_pred)
save_doc_figure(confusion_figure, "combined_cnn_confusion", MODEL_DOCS_FIGURES_DIR)

recall_figure = plot_class_recall(result.y_val, result.y_pred)
save_doc_figure(recall_figure, "combined_cnn_recall", MODEL_DOCS_FIGURES_DIR)

confusions_figure = plot_top_confusions(result.y_val, result.y_pred)
save_doc_figure(confusions_figure, "combined_cnn_confusions", MODEL_DOCS_FIGURES_DIR)

{
    "tuning_run": tuned.run_id,
    "evaluation_run": result.run_id,
    "invalid_combinations": result.metrics.invalid_combinations,
}